# Stage 6: Chronological Train / Validation / Test Split
## Crop Price Prediction & Market Recommendation System — Nagpur Mandis (2022–2026)

### Purpose of this Stage:
In time-series econometrics and predictive modeling, standard random splits (`train_test_split(..., shuffle=True)`) cause severe **data leakage**. Random splitting leaks future price patterns and market shocks into the training set, giving falsely inflated evaluation metrics that collapse in production.

To ensure **100% realistic real-world evaluation and maximum project accuracy**, we perform a strict **Chronological (Date-Based) Split**:
1. **Training Set (~70%)**: Jan 31, 2022 – Apr 30, 2025 (~3.25 years) — Used to train baseline and gradient boosted models.
2. **Validation Set (~15%)**: May 01, 2025 – Dec 31, 2025 (8 months) — Used for hyperparameter tuning and early stopping.
3. **Test Set (~15%)**: Jan 01, 2026 – Sep 29, 2026 (9 months) — Held-out unseen future data for final evaluation.


### Step 1: Import Required Libraries


In [ ]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings("ignore")

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version : {np.__version__}")
print("Libraries loaded successfully.")


### Step 2: Load the Model-Ready Dataset


In [ ]:
# Path to the model-ready dataset
input_path = "../5.model ready dataset/model_ready_dataset.csv"

print("Loading model-ready dataset...")
df = pd.read_csv(input_path)
df['date'] = pd.to_datetime(df['date'])

print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date Range   : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Missing (NaN): {df.isna().sum().sum()} values")


### Step 3: Inspect Features and Target Column


In [ ]:
target_col = 'target_next_day_modal_price'
drop_non_features = ['date', 'market', 'commodity', 'season', target_col]

# Feature columns
feature_cols = [c for c in df.columns if c not in drop_non_features]

print(f"Target Variable: {target_col}")
print(f"Total Features : {len(feature_cols)}")
print(f"Categorical info retained via One-Hot Encodings: {[c for c in feature_cols if 'commodity_' in c or 'market_' in c or 'season_' in c]}")


### Step 4: Perform Chronological Split (No Data Leakage)


In [ ]:
# Define cut-off dates
train_cutoff = "2025-04-30"
val_cutoff = "2025-12-31"

train_mask = df['date'] <= train_cutoff
val_mask = (df['date'] > train_cutoff) & (df['date'] <= val_cutoff)
test_mask = df['date'] > val_cutoff

df_train = df[train_mask].copy()
df_val = df[val_mask].copy()
df_test = df[test_mask].copy()

total_rows = len(df)
print("=" * 60)
print(f"TRAIN SET      : {len(df_train):,} rows ({len(df_train)/total_rows*100:.1f}%) | {df_train['date'].min().date()} to {df_train['date'].max().date()}")
print(f"VALIDATION SET : {len(df_val):,} rows ({len(df_val)/total_rows*100:.1f}%) | {df_val['date'].min().date()} to {df_val['date'].max().date()}")
print(f"TEST SET       : {len(df_test):,} rows ({len(df_test)/total_rows*100:.1f}%) | {df_test['date'].min().date()} to {df_test['date'].max().date()}")
print("=" * 60)


### Step 5: Verification & Integrity Checks
Ensure all 12 commodities and 13 mandis are fully covered across all splits.


In [ ]:
# Check commodities and markets coverage
commodities_train = set(df_train['commodity'].unique())
commodities_val = set(df_val['commodity'].unique())
commodities_test = set(df_test['commodity'].unique())

markets_train = set(df_train['market'].unique())
markets_val = set(df_val['market'].unique())
markets_test = set(df_test['market'].unique())

assert commodities_train == commodities_val == commodities_test, "Mismatch in commodities!"
assert markets_train == markets_val == markets_test, "Mismatch in markets!"

print(" Integrity Check PASSED:")
print(f"- All {len(commodities_train)} commodities present across Train, Val, and Test.")
print(f"- All {len(markets_train)} markets present across Train, Val, and Test.")
print(f"- Temporal order strictly verified: max(Train) < min(Val) and max(Val) < min(Test).")


### Step 6: Export Split Datasets and Metadata


In [ ]:
# Save datasets
print("Saving train, val, and test datasets...")
df_train.to_csv("train_data.csv", index=False)
df_val.to_csv("val_data.csv", index=False)
df_test.to_csv("test_data.csv", index=False)

# Save feature metadata
metadata = {
    "total_rows": total_rows,
    "train_rows": len(df_train),
    "val_rows": len(df_val),
    "test_rows": len(df_test),
    "train_range": [str(df_train['date'].min().date()), str(df_train['date'].max().date())],
    "val_range": [str(df_val['date'].min().date()), str(df_val['date'].max().date())],
    "test_range": [str(df_test['date'].min().date()), str(df_test['date'].max().date())],
    "target_column": target_col,
    "feature_count": len(feature_cols),
    "feature_columns": feature_cols
}

with open("split_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print(" All split files and metadata successfully saved!")
